In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Agent Retrieval における Target Recall を使った再現率とレイテンシのチューニング

このノートブックでは、[Gemini Enterprise Agent Platform](https://docs.cloud.google.com/gemini-enterprise-agent-platform) の **[Agent Retrieval](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/overview)**（旧 [Vector Search 2.0](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/overview)）の検索パラメータ **`target_recall`** が、300 万件のインデックスにおいて検索結果の品質（再現率）とクエリ レイテンシにどのような影響を与えるかを実測・解説します。

Google Cloud プロジェクトがなくても実行できます。すべてのクエリは公開されている
[Agent Retrieval インタラクティブ デモ](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/try-it)のエンドポイントに送信されるため、リソースの作成や待機、クリーンアップ作業は一切不要です。

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall_ja.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Colab で開く
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fembeddings%2Fagent-retrieval-target-recall_ja.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Colab Enterprise で開く
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/embeddings/agent-retrieval-target-recall_ja.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Workbench で開く
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall_ja.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> GitHub で表示
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>共有:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall_ja.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall_ja.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall_ja.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall_ja.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-target-recall_ja.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>


| 著者 |
| --- |
| [Kaz Sato](https://github.com/kazunori279) |


## 概要

### 通常はデフォルト設定のままで十分です

Agent Retrieval は、すべての[近似最近傍（ANN: Approximate Nearest Neighbor）](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/overview)探索において、再現率とレイテンシのバランスを自動的に最適化します。`target_recall` を指定しない場合、インデックスの構造に基づいて高品質かつ高速な検索設定が自動選択されます。大半のアプリケーションではこのデフォルト設定が最適であり、基盤エンジンの改良に伴い自動的に改善されていきます。

`target_recall` は、この再現率と速度のトレードオフをユーザー自身で直接制御したい場合に使用します。

### target_recall の仕組み

希望する目標再現率を `0.0` から `1.0` の間の数値で指定します。

- **`0.95`** - 全件探索（ブルートフォース探索）で見つかるはずの最近傍結果のうち、約 95% を返すようインデックスを走査します。

このパラメータが登場する前は、[`search_leaves_pct` や `initial_candidate_count`](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/configuring-indexes) といった内部実装レベルの設定を手動調整する必要があり、インデックスの内部構造に関する深い知識が求められていました。`target_recall` ではユーザーが実際に重視する「目標再現率」という単位で意図を直接指定でき、サービス側がインデックス構築時に収集した測定データに基づいて内部の走査パラメータへ自動マッピングします。

なお、再現率はベストエフォートです。実際に得られる再現率はデータセットの内容や個々のクエリによって異なるため、このノートブックの Part 3 では机上の理論値ではなく実測値で検証します。

### チューニングすべきケース

| 状況 | 推奨アクション |
|---|---|
| デフォルト以上の高い検索品質・網羅性を確保したい | `target_recall` を `1.0` に近づける |
| わずかな再現率の向上よりも、レイテンシ削減やスループット（QPS）向上を優先したい | `target_recall` を下げて検索の走査コストを抑える |
| デフォルトの挙動・速度で満足している | `target_recall` を設定しない（未指定のままにする） |

### 適用条件: クラスタ化された ANN インデックス

`target_recall` はクラスタ化された [ANN インデックス](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/indexes/indexes)に対して有効です。Agent Retrieval は[コレクション (Collection)](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/collections/collections) の埋め込みベクトル数が約 **100,000 件** を超えると ANN インデックスを構築します。10 万件未満の小さなコレクションでは全件総当たりのブルートフォース探索が行われるため、常に再現率 100% の完全一致探索となり、トレードオフを行う余地がありません（小規模コレクションでパラメータを指定してもエラーにはならず、単に無視されます）。

`target_recall` は**パフォーマンス最適化 (performance-optimized)** および**ストレージ最適化 (storage-optimized)** の両方のインデックス ティアでサポートされています。なお、再現率は常にベストエフォートであり、完全な保証ではありません。

そのため、本ノートブックでは 300 万件のマルチモーダル インデックスを使用してこれらの動作を検証します。

### ノートブックの流れ

1. 公開デモエンドポイントへの接続（GCP プロジェクトや認証設定は不要）
2. 1 つのクエリで `target_recall` の値を変更し、検索結果の変化を確認
3. 多数のクエリで再現率とレイテンシを実測し、トレードオフ曲線をプロット
4. 自身のプロジェクトで利用するための REST API リクエスト形式の確認


---

# Part 1: デモ エンドポイントへの接続

必要なライブラリは `requests` のみです（グラフ描画と集計用に `pandas` と `matplotlib` を使用）。認証設定や API キーは不要で、そのまま利用できます。


In [ ]:
%pip install --upgrade --quiet requests pandas matplotlib tqdm

print("準備完了。")


このエンドポイントは、[Agent Retrieval インタラクティブ デモ](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/try-it) ページで提供されている Mercari の商品データセットを検索します。ここでは **マルチモーダル (multimodal)** データセットを使用します。

- **300 万件のアイテム:** ANN インデックスが確実に構築される規模です。
- **クエリの埋め込み生成とベクトル検索が分離:** エンドポイントはクエリの埋め込み生成時間とインデックス検索時間を個別に返します。この分離は極めて重要です。クエリの埋め込み生成には数百ミリ秒かかるのに対し、ベクトル検索自体は数十ミリ秒で完了します。エンドツーエンドの合計時間で計測してしまうと、検索処理のレイテンシ変化が埋め込み生成時間のばらつきに埋もれてしまいます。

以降のすべての測定結果では、クエリ埋め込み生成時間やネットワーク転送時間を除外した、**サーバー側のベクトル検索時間のみ**を評価します。


In [ ]:
from typing import Any

import requests

DEMO_ENDPOINT = "https://ac-web2-nhhfh7g7iq-uc.a.run.app/api/query"
DATASET_ID = "mercari3m_multimodal"  # @param {type:"string"}
TOP_K = 10


def search(
    query: str,
    target_recall: float | None = None,
    top_k: int = TOP_K,
    dataset_id: str = DATASET_ID,
) -> dict[str, Any]:
    """デモ エンドポイントに対して 1 回の検索を実行します。

    引数:
      query: 自然言語の検索クエリ。
      target_recall: 希望する目標再現率（0.0 〜 1.0）。None の場合はサービスが自動選択。
      top_k: 取得する結果件数。
      dataset_id: 検索対象のデモ データセット。

    戻り値:
      "ids", "names", "search_ms" (クエリ埋め込み生成とネットワークを除いたサーバー側検索時間)、
      "embed_ms" (クエリ埋め込み生成時間) を含む辞書。
    """
    payload: dict[str, Any] = {
        "query": query,
        "dataset_id": dataset_id,
        "use_semantic_search": True,
        "use_text_search": False,
        "rows": top_k,
    }
    if target_recall is not None:
        if not 0.0 <= target_recall <= 1.0:
            raise ValueError("target_recall は 0.0 から 1.0 の間で指定してください。")
        payload["target_recall"] = target_recall

    response = requests.post(DEMO_ENDPOINT, json=payload, timeout=120)
    if response.status_code != 200:
        raise RuntimeError(
            f"検索に失敗しました ({response.status_code}): {response.text[:300]}"
        )
    body = response.json()

    return {
        "ids": [item["id"] for item in body["items"]],
        "names": [item.get("name", "") for item in body["items"]],
        # サーバー側で計測された所要時間（ネットワーク レイテンシを除く）
        "search_ms": body["latencies"]["query"] * 1000,
        "embed_ms": body["latencies"]["gen_query_emb"] * 1000,
        "applied_target_recall": body.get("applied_target_recall"),
    }


# エンドポイントの疎通確認と target_recall の対応を確認
probe = search("camera", target_recall=0.9, top_k=3)
if probe["applied_target_recall"] != 0.9:
    raise RuntimeError(
        "デモ エンドポイントが target_recall を認識しませんでした。"
        "パラメータ追加前の古いビルドが動作している可能性があります。"
    )
print(f"接続成功。検索結果の例: {probe['names'][0]!r}")
print(
    f"  クエリ埋め込み生成: {probe['embed_ms']:6.1f} ms  (以下の測定からは除外)"
)
print(f"  インデックス検索:   {probe['search_ms']:6.1f} ms  (測定対象)")


---

# Part 2: target_recall による検索結果の変化を確認

最も低い設定ではインデックスの走査を最小限に抑え、最も高い設定では徹底的な探索を行います。同じインデックス、同じクエリに対して、パラメータのみを変更して比較します。

なお、単発のクエリではネットワークやキャッシュのブレが生じるため、ここではレイテンシは表示しません。正確な所要時間は Part 3 で測定します。


In [ ]:
EXAMPLE_QUERY = "vintage leather camera bag"  # @param {type:"string"}

reference = search(EXAMPLE_QUERY, target_recall=1.0, top_k=5)
print(f"クエリ: {EXAMPLE_QUERY!r}\n")
print("target_recall=1.0 (最も網羅的な探索):")
for rank, name in enumerate(reference["names"], 1):
    print(f"  {rank}. {name[:70]}")

for recall_target in [0.1, 0.5, 0.9]:
    result = search(EXAMPLE_QUERY, target_recall=recall_target, top_k=5)
    overlap = len(set(result["ids"]) & set(reference["ids"]))
    print(
        f"\ntarget_recall={recall_target} (網羅的探索結果のうち {overlap}/5 件が一致):"
    )
    for rank, name in enumerate(result["names"], 1):
        marker = " " if result["ids"][rank - 1] in reference["ids"] else "*"
        print(f"  {rank}.{marker}{name[:70]}")

print("\n* = target_recall=1.0 の結果に含まれていないアイテム")


---

# Part 3: 再現率とレイテンシの実測

### ここでの再現率の測定方法

再現率は、完全な全件探索（正解の最近傍）と比較して評価します。ただし、300 万件のインデックスではブルートフォース探索（完全一致 kNN）を直接実行することはできません（ブルートフォース探索は 10 万件未満のインデックスでのみ提供されるため）。そこで、インデックスが提供できる最も徹底的な探索である **`target_recall = 1.0` の結果をリファレンス（基準）** とし、各設定でどれだけのアイテムが保持されるかを測定します。

$$\text{Recall@K} \approx \frac{|\text{設定 } t \text{ の結果} \;\cap\; \text{設定 } 1.0 \text{ の結果}|}{K}$$

この値は、最も網羅的な ANN 探索に対する相対的な再現率を示しています。ご自身のプロジェクトで生の埋め込みベクトルを保持している場合は、オフラインで完全な Ground Truth（真の最近傍）を計算して検証することも可能です。

### 測定結果の信頼性を担保する工夫

- **非線形領域（0.9〜1.0）の細密サンプリング:** 探索コストとレイテンシは 0.9 から 1.0 の間で非線形に急上昇するため、低〜中域は大まかに測定しつつ（`0.1`, `0.5`, `0.8`）、高再現率領域（`0.90`, `0.92`, `0.94`, `0.96`, `0.98`, `0.99`, `1.0`）を細かく刻んでサンプリングします。
- **クエリ実行順序のシャッフル:** 目標値を低い順から高い順へ順次実行すると、後半のクエリがインデックスのキャッシュが効いた状態で測定されてしまい、誤ったレイテンシの傾向が生じます。実行順序をランダムにシャッフルすることでキャッシュによる偏りを防ぎます。
- **ウォームアップの実施と中央値の採用:** 初回リクエストをウォームアップとして除外し、複数回測定した中央値（Median）を採用することで、単発の突発的な遅延の影響を排除します。


In [ ]:
import random
import statistics

import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

QUERIES = [
    "vintage leather camera bag",
    "running shoes",
    "coffee mug",
    "wool sweater",
    "gold necklace",
    "gaming keyboard",
]
# 非線形な急上昇カーブを捉えるため、0.9〜1.0 を細密にサンプリング
RECALL_TARGETS = [0.1, 0.5, 0.8, 0.9, 0.92, 0.94, 0.96, 0.98, 0.99, 1.0]
REPEATS = 3

# 1. 各クエリのリファレンス結果（最も徹底的な探索結果）を作成
print(f"{len(QUERIES)} 件のクエリに対するリファレンス結果を作成中...")
reference_ids = {
    query: set(search(query, target_recall=1.0)["ids"])
    for query in tqdm(QUERIES, desc="リファレンス作成")
}

# 2. 初回測定のオーバーヘッドを除外するためのウォームアップ
search(QUERIES[0], target_recall=0.9)

# 3. キャッシュの偏りを排除するために (クエリ, 目標値) のペアをシャッフル
plan = [(q, t) for q in QUERIES for t in RECALL_TARGETS for _ in range(REPEATS)]
random.seed(42)
random.shuffle(plan)

latencies: dict[float, list[float]] = {t: [] for t in RECALL_TARGETS}
recalls: dict[float, list[float]] = {t: [] for t in RECALL_TARGETS}
embed_times: list[float] = []
scored: set[tuple[str, float]] = set()

print(f"{len(plan)} 件のクエリ測定を実行中...")
for query, recall_target in tqdm(plan, desc="クエリ測定"):
    result = search(query, target_recall=recall_target)
    latencies[recall_target].append(result["search_ms"])
    embed_times.append(result["embed_ms"])

    # 再現率の計算は各 (クエリ, 目標値) につき 1 回のみ実施（繰り返しはレイテンシ測定用）
    if (query, recall_target) in scored:
        continue
    scored.add((query, recall_target))
    kept = set(result["ids"]) & reference_ids[query]
    recalls[recall_target].append(len(kept) / TOP_K)

df_metrics = pd.DataFrame(
    [
        {
            "target_recall": t,
            "achieved_recall": statistics.mean(recalls[t]),
            "median_search_ms": statistics.median(latencies[t]),
        }
        for t in RECALL_TARGETS
    ]
)

median_embed_ms = statistics.median(embed_times)
search_lo = df_metrics["median_search_ms"].min()
search_hi = df_metrics["median_search_ms"].max()
print()
print(
    f"クエリ埋め込み生成: 1回あたり中央値 {median_embed_ms:.0f} ms (下表からは除外)"
)
print(f"インデックス検索自体: 各設定で {search_lo:.0f}〜{search_hi:.0f} ms")
print("エンドツーエンドで測定した場合、埋め込み生成時間に隠れてこの差は見えなくなります。")
df_metrics


以下のグラフは、検索コスト（検索時間の中央値 ms）と検索結果の品質（実測再現率 Recall@K）の関係を示したものです。


In [ ]:
fig, (ax_latency, ax_recall) = plt.subplots(1, 2, figsize=(12, 4))

ax_recall.plot(
    df_metrics["target_recall"],
    df_metrics["achieved_recall"],
    marker="s",
    color="#34a853",
    label="Achieved",
)
ax_recall.plot([0, 1], [0, 1], linestyle="--", color="#9aa0a6", label="Requested")
ax_recall.set_xlabel("Requested target_recall")
ax_recall.set_ylabel(f"Achieved recall@{TOP_K}")
ax_recall.set_title("Higher target_recall achieves higher recall")
ax_recall.set_ylim(0, 1.05)
ax_recall.legend()
ax_recall.grid(True, linestyle="--", alpha=0.5)

ax_latency.plot(
    df_metrics["target_recall"],
    df_metrics["median_search_ms"],
    marker="o",
    color="#1a73e8",
)
ax_latency.set_xlabel("Requested target_recall")
ax_latency.set_ylabel("Median search time (ms)")
ax_latency.set_title("Higher recall requires more search time")
ax_latency.set_ylim(bottom=0)
ax_latency.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


## 結果の分析

**トレードオフは実在し、非線形です。** 検索時間は目標設定が中程度の間はほぼ横ばいですが、上限付近で急激に跳ね上がります。再現率の最後の数パーセントを拾い上げるために、インデックスは大幅に多くのリーフや候補ベクトルを探索する必要があるためです。たとえば、`target_recall` を `0.99` から `0.90` に下げると検索時間は約半分に短縮されますが、`0.90` から `0.50` に下げてもそれ以上の短縮効果はごくわずかです。この曲線の形状を把握することで、ワークロードにとって最も効率的なチューニングポイントが見えてきます。

**再現率は指定に応じて向上し、一定の最低値で底打ちします。** ある閾値を下回るとインデックスが探索する最小限の仕事量に達するため、設定をそれ以上下げても返される結果は変わらなくなります。また、実測再現率が指定した `target_recall` を上回ることがよくありますが、これはサービスが指定された目標を達成するために安全側の設定を選択するためです。

**全体の所要時間を支配しているのは検索ではなく埋め込み生成です。** クエリ埋め込みの生成には 1 回あたり数百ミリ秒を要するのに対し、ベクトル検索自体は数十ミリ秒で完了しています。もしエンドツーエンドの時間で測定していた場合、上記のレイテンシ短縮効果は埋もれて見えなくなっていたはずです。ご自身のデータでベンチマークを行う際も、埋め込み生成時間と検索時間を明確に分離することが重要です。


---

# Part 4: 自身のプロジェクトでの利用方法

デモ エンドポイントは挙動を検証するためのものです。本番アプリケーションでは Agent Retrieval の [Vector Search REST API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/query-search/search) を直接呼び出し、インデックスを指定する `indexHint` 配下の `denseScannParams` に `targetRecall` を指定します。

```json
{
  "vectorSearch": {
    "searchField": "product_embedding",
    "vector": { "values": [0.0245, -0.0408, "..."] },
    "topK": 10,
    "searchHint": {
      "indexHint": {
        "name": "projects/PROJECT/locations/LOCATION/collections/COLLECTION/indexes/INDEX",
        "denseScannParams": {
          "targetRecall": 0.95
        }
      }
    }
  }
}
```

エンドポイント (詳細は [データ オブジェクトの検索](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/query-search/search) を参照):

```
POST https://vectorsearch.googleapis.com/v1/projects/{project}/locations/{location}/collections/{collection}/dataObjects:search
```

### Python による REST API 呼び出しの例

下のセルでは、`google.auth` と `requests` を使用して `targetRecall` を指定した検索リクエストを送信するサンプルコードを示します。

### 実装時のポイント

- **リクエストごとに設定可能:** インデックスの再構築や再デプロイを行うことなく、クエリごとに動的に `target_recall` を変更できます（例: 最初の粗い絞り込みは `0.7`、高精度が求められるランキング処理は `0.99` など）。
- **`indexHint` が必要:** `targetRecall` は `searchHint.indexHint.denseScannParams` 配下に指定します。ヒントを指定しないリクエストではシステムのデフォルト設定が使われ、`knnHint` を指定した場合はブルートフォース探索が行われます。
- **データ オブジェクトの検索:** ベクトル検索はコレクション内に格納された [データ オブジェクト (Data Objects)](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/data-objects/data-objects) に対して実行されます。
- **REST API の利用:** クライアント ライブラリ（例: `google-cloud-vectorsearch` 0.11.3 時点）では、`DenseScannParams` にまだ `target_recall` が含まれていない場合があります。その場合は上記のように REST API を直接呼び出します。
- **API バージョンによる違い:** `v1` では `denseScannParams` のパラメータは `targetRecall` が基本です。旧来のパラメータ（`searchLeavesPct` や `initialCandidateCount`）は `v1beta` にのみ存在し、`targetRecall` と併用することはできません。`v1` のリクエストに `searchLeavesPct` を指定すると `Unknown name "searchLeavesPct"` エラーになります。
- **インデックス ティアの互換性:** **パフォーマンス最適化**および**ストレージ最適化**の両方のインデックス ティアでサポートされています。なお、再現率は常にベストエフォートです。


In [ ]:
import json

# ==============================================================================
# サンプル: targetRecall を指定して Vector Search REST API を呼び出す
#
# このコードは REST リクエストのペイロード構造と呼び出し手順を示しています。
# 自身の GCP プロジェクトに対して実際に実行する場合は、以下の手順で行ってください:
#
# 1. Colab で実行している場合は、最初に認証を実行します:
#      from google.colab import auth
#      auth.authenticate_user()
#
# 2. ローカル環境で実行している場合:
#      gcloud auth application-default login
#
# 3. 下記のプレースホルダーをご自身のプロジェクトやコレクションの情報に置き換えます。
# ==============================================================================

# 1. 設定 - ご自身の Google Cloud プロジェクトの情報に置き換えてください
PROJECT_ID = "YOUR_PROJECT_ID"
LOCATION = "us-central1"
COLLECTION_ID = "YOUR_COLLECTION_ID"
INDEX_ID = "YOUR_INDEX_ID"
SEARCH_FIELD = "product_embedding"

# クエリ ベクトルの例（実際の埋め込みベクトルに置き換えてください）
QUERY_VECTOR = [0.0245, -0.0408, 0.0123]

# 2. API エンドポイントと targetRecall を含むリクエスト ペイロードの構築
collection_path = (
    f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}"
)
index_path = f"{collection_path}/indexes/{INDEX_ID}"
endpoint_url = (
    f"https://vectorsearch.googleapis.com/v1/{collection_path}/dataObjects:search"
)

payload = {
    "vectorSearch": {
        "searchField": SEARCH_FIELD,
        "vector": {"values": QUERY_VECTOR},
        "topK": 10,
        "searchHint": {
            "indexHint": {
                "name": index_path,
                "denseScannParams": {
                    "targetRecall": 0.95,
                },
            }
        },
    }
}

print("構築されたリクエスト URL:")
print(f"  {endpoint_url}\n")
print("構築されたリクエスト ペイロード:")
print(json.dumps(payload, indent=2))

# ==============================================================================
# 3. リクエスト送信（認証済みの GCP 環境で実行する場合はコメントを解除してください）
# ==============================================================================
# import google.auth
# from google.auth.transport.requests import Request
# import requests
#
# credentials, _ = google.auth.default(
#     scopes=["https://www.googleapis.com/auth/cloud-platform"]
# )
# credentials.refresh(Request())
# headers = {
#     "Authorization": f"Bearer {credentials.token}",
#     "Content-Type": "application/json",
# }
#
# response = requests.post(endpoint_url, json=payload, headers=headers)
# response.raise_for_status()
# search_results = response.json()
# print("検索結果:", search_results)


---

# まとめ

- **自動バランス:** Agent Retrieval はすべての ANN 探索において再現率とレイテンシを自動で最適化します。多くのケースでは `target_recall` を未指定のまま利用するのが推奨されます。
- **きめ細かな制御:** より高い検索品質が必要な場合は `1.0` に近づけ、再現率をわずかに犠牲にしてでもレイテンシや QPS を改善したい場合は値を下げることで、意図に応じたチューニングが可能です。
- **直感的な抽象化:** `search_leaves_pct` や `initial_candidate_count` などの低レベル設定を意識することなく、「目標とする再現率」という分かりやすい指標で調整できます。
- **クエリ単位の柔軟性:** `searchHint.indexHint.denseScannParams.targetRecall` を通じて、リクエストごとに動的に適用できます。
- **クラスタ化 ANN インデックスで有効:** 約 10 万件以上の埋め込みベクトルを持つコレクションに適用されます。それ以下の小規模なコレクションではブルートフォース探索（常に再現率 100%）となるため、本パラメータは機能しません。
- **インデックス ティアのサポート:** パフォーマンス最適化およびストレージ最適化の両ティアでベストエフォートとしてサポートされています。
- **ベンチマーク時の注意点:** 自社データで検証する際は、クエリ埋め込み生成時間とインデックス検索時間を必ず分離してください（埋め込み生成時間が検索時間を圧倒的に上回るため）。また、クエリ実行順序のシャッフルやウォームアップ、中央値の採用を行ってください。

## 次のステップ

- **[Agent Retrieval インタラクティブ デモ](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/try-it)**: 他のデータセットや検索タイプをブラウザ上で体験できます。
- **[Agent Retrieval 概要](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/overview)**: コレクション、データ オブジェクト、インデックスの概念について確認します。
- **[コレクションのドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/collections/collections)**: コレクションの作成と構成について確認します。
- **[データ オブジェクトのドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/data-objects/data-objects)**: データ オブジェクトの取り込み、更新、管理について確認します。
- **[インデックスのドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/indexes/indexes)**: ベクトル検索用インデックスの管理について確認します。
- **[データ オブジェクトの検索](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/query-search/search)**: セマンティック検索、テキスト検索、ハイブリッド検索、ベクトル検索のドキュメントです。
- **[Agent Development Kit (ADK)](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/adk)**: AI エージェントの構築とデプロイのためのフレームワークです。
- **[Agent Retrieval と ADK で旅行エージェントを構築する](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/vector-search-2-travel-agent.ipynb)**: ハイブリッド検索を AI エージェントのツールとして組み込む実装例です。
